# Document Intelligence with Hugging Face Transformers

## 💎 Case Overview

You are an AI Engineer at a legal tech startup. Your firm processes massive, dense documents every day. Lawyers spend hours reading through these documents just to find specific clauses, summarize paragraphs, or find related case files.

Your objective is to build an automated Document Intelligence pipeline that can:

1. Read a paragraph and answer specific questions about it (Extractive QA).
2. Condense long-winded legal and factual text into short summaries.
3. Convert text into mathematical vectors (embeddings) to measure how semantically similar two sentences are.

### 🐢 Your Dataset

- We will use a small subset of the `squad` (Stanford Question Answering Dataset) to simulate factual documents.

### ⚔️ Your Goal

Complete the following Deep Learning tasks:

- 👉 Build an Extractive Question Answering function.
- 👉 Build a Text Summarization function.
- 👉 Generate sentence embeddings and calculate Cosine Similarity between texts.


---

▶️ Run the code cell below to import `unittest`, a module used for **🧭 Check Your Work** sections and the autograder.


In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import base64
import unittest

tc = unittest.TestCase()

---

### 🔨 Import Packages and Dataset

▶️ Run the code cell below to install packages and load the necessary libraries.
_(Note: Google Colab comes with many of these, but we need to ensure `sentence-transformers` is installed)._


In [ ]:
!pip install transformers datasets sentence-transformers scipy pandas

In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine

print("Loading SQuAD dataset slice...")
dataset = load_dataset("squad", split="validation[:10]")
df = pd.DataFrame(dataset)

print(f"Dataset loaded with {len(df)} records.")
df[["context", "question", "answers"]].head(3)

---

## 🤖 Part 1: Information Extraction


---

### 🎯 Deliverable 1: Extractive Question Answering

Extractive QA models read a `context` and a `question`, and extract the exact span of text from the context that answers the question.

#### 👇 Tasks

- ✔️ Instantiate a `pipeline` for `"question-answering"` and store it in `qa_pipeline`. (You can let Hugging Face choose the default model).
- ✔️ Complete the function `answer_question(context, question)` to return the extracted answer string.


In [ ]:
print("Loading QA Model...")
# YOUR CODE BEGINS
qa_pipeline = pipeline("question-answering")


def answer_question(context, question):
    result = qa_pipeline(question=question, context=context)
    return result["answer"]


# YOUR CODE ENDS

#### 🧭 Check Your Work


In [ ]:
_test_case = "deliverable-01"
_points = 30

test_context = "The Python programming language was created by Guido van Rossum and first released in 1991."
test_question = "Who created Python?"
ans = answer_question(test_context, test_question)

tc.assertTrue("Guido van Rossum" in ans, "Failed: Did not extract the correct creator.")
print("Task 1 (Question Answering) Passed!")

---

### 🎯 Deliverable 2: Text Summarization

#### 👇 Tasks

- ✔️ Instantiate a `pipeline` for `"summarization"` using the model `"sshleifer/distilbart-cnn-12-6"` (a lightweight, fast summarizer) and store it in `sum_pipeline`.
- ✔️ Complete the function `summarize_text(text)` to return the summary text. Set `max_length=50` and `min_length=10` in the pipeline call.


In [ ]:
print("Loading Summarization Model...")
# YOUR CODE BEGINS
sum_pipeline = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")


def summarize_text(text):
    result = sum_pipeline(text, max_length=50, min_length=10, truncation=True)
    return result[0]["summary_text"]


# YOUR CODE ENDS

#### 🧭 Check Your Work


In [ ]:
_test_case = "deliverable-02"
_points = 35

long_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans. 
Leading AI textbooks define the field as the study of intelligent agents: any system that perceives its environment and takes actions that maximize its chance of achieving its goals. 
Some popular accounts use the term artificial intelligence to describe machines that mimic cognitive functions that humans associate with the human mind, such as learning and problem solving.
"""
summary = summarize_text(long_text)

tc.assertTrue(
    len(summary) < len(long_text),
    "Failed: Summary should be shorter than the original text.",
)
tc.assertTrue(isinstance(summary, str), "Failed: Function must return a string.")
print("Task 2 (Summarization) Passed!")

---

## 📐 Part 2: Semantic Similarity with Embeddings


---

### 🎯 Deliverable 3: Compute Sentence Similarity

Instead of looking for exact keyword matches, modern NLP converts text into dense vector embeddings. Texts with similar meanings will have vectors pointing in similar directions.

#### 👇 Tasks

- ✔️ Instantiate a `SentenceTransformer` model using `"all-MiniLM-L6-v2"` and store it in `embedder`.
- ✔️ Complete the function `calculate_similarity(sentence1, sentence2)`.
- ✔️ Use `embedder.encode()` to get vectors for both sentences.
- ✔️ Use `scipy.spatial.distance.cosine` to find the cosine distance, and return the cosine similarity (`1 - distance`).


In [ ]:
print("Loading Embedding Model...")
# YOUR CODE BEGINS
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def calculate_similarity(sentence1, sentence2):
    vec1 = embedder.encode(sentence1)
    vec2 = embedder.encode(sentence2)

    # Cosine similarity is 1 minus the cosine distance
    similarity = 1 - cosine(vec1, vec2)
    return float(similarity)


# YOUR CODE ENDS

#### 🧭 Check Your Work


In [ ]:
_test_case = "deliverable-03"
_points = 35

s1 = "The cat rests on the rug."
s2 = "A kitten is sleeping on the carpet."
s3 = "I am writing Python code for data science."

sim_high = calculate_similarity(s1, s2)
sim_low = calculate_similarity(s1, s3)

tc.assertTrue(
    sim_high > 0.6,
    "Failed: The semantic similarity between the cat sentences should be high.",
)
tc.assertTrue(
    sim_low < 0.3,
    "Failed: The semantic similarity with the coding sentence should be low.",
)
tc.assertTrue(
    sim_high > sim_low,
    "Failed: Similar sentences must score higher than unrelated ones.",
)
print("Task 3 (Semantic Similarity) Passed!")